## STADIOchoice Churn Prediction

### Objective and problem definition

Objective: Test whether subscriber/account information and behaviour available at a monthly index date can rank subscribers by the probability of a qualifying churn event in the following **30 days**.

The supplied data dictionary defines churn as subscriber-initiated cancellation without reactivation within 30 days, and the supplied `churn_indicator` is the authoritative outcome flag. A 30-day prediction horizon is therefore distinct from the 30-day reactivation rule.

Observation grain: One eligible subscriber per month-end index date. A subscriber may appear in multiple periods because the intended use is forecasting future behaviour for existing subscribers. Features use only records dated on or before the index date; the label uses the following 30 days. The final outcome table and final subscription status are never predictors.

### Environment Setup and Imports

In [2]:

from pathlib import Path
import sys
import platform
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.base import clone



SEED = 42
HORIZON_DAYS = 30
LOOKBACK_DAYS = 90

CURRENT_DIR = Path.cwd().resolve()

possible_roots = [CURRENT_DIR, *CURRENT_DIR.parents]

PROJECT_ROOT = next(
    (
        root
        for root in possible_roots
        if (root / "data" / "stadiochoice").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project data directory.\n"
        f"Current working directory: {CURRENT_DIR}\n"
        "Expected: <project-root>/data/stadiochoice/"
    )

DATA_DIR = PROJECT_ROOT / "data" / "stadiochoice"

FILES = {
    "account": "2_subscriber_account.csv",
    "subscription": "3_subscription.csv",
    "viewing": "4_viewing_behaviour.csv",
    "engagement": "5_engagement_behaviour.csv",
    "content": "5b_content_reference.csv",
    "billing": "6_billing_payment_history.csv",
    "outcome": "7_cancellation_retention.csv",
}

print(
    f"Python: {platform.python_version()} | "
    f"pandas: {pd.__version__} | "
    f"NumPy: {np.__version__}"
)
print(f"Project root: {PROJECT_ROOT}")
print(f"Input directory: {DATA_DIR}")

missing_files = [
    filename
    for filename in FILES.values()
    if not (DATA_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files in {DATA_DIR}:\n"
        + "\n".join(missing_files)
    )

print(f"All {len(FILES)} required CSV files are present.")

Python: 3.13.7 | pandas: 3.0.5 | NumPy: 2.5.2
Project root: C:\Users\sltha\Documents\STADIO\CAP182\capstone-project
Input directory: C:\Users\sltha\Documents\STADIO\CAP182\capstone-project\data\stadiochoice
All 7 required CSV files are present.


### Data inventory and loading

The README describes seven operational tables and their intended grains. We load them without silently dropping records, parse timestamp fields explicitly, and compare actual shapes with the documented counts. The content lookup enriches viewing records, but the churn modelling table will be built at subscriber-month grain.

In [3]:
tables = {
    name: pd.read_csv(DATA_DIR / filename)
    for name, filename in FILES.items()
}

date_cols = {
    "account": [
        "account_creation_date",
        "acquisition_date",
    ],
    "subscription": [
        "subscription_start_date",
        "subscription_end_date",
    ],
    "viewing": [
        "viewing_timestamp",
    ],
    "engagement": [
        "event_timestamp",
    ],
    "billing": [
        "billing_date",
    ],
    "outcome": [
        "cancellation_date",
        "reactivation_date",
        "retention_interaction_date",
    ],
}

for table, columns in date_cols.items():
    for column in columns:
        tables[table][column] = pd.to_datetime(
            tables[table][column],
            errors="coerce",
        )


table_grains = {
    "account": "one row per subscriber",
    "subscription": "one row per subscription",
    "viewing": "one row per viewing session",
    "engagement": "one row per engagement event",
    "content": "one row per content item",
    "billing": "one row per billing event",
    "outcome": "one row per cancellation/retention event",
}

inventory = pd.DataFrame([
    {
        "table": name,
        "file": FILES[name],
        "rows": len(df),
        "columns": df.shape[1],
        "subscriber_ids": (
            df["subscriber_id"].nunique()
            if "subscriber_id" in df.columns
            else np.nan
        ),
        "grain_from_documentation": table_grains[name],
    }
    for name, df in tables.items()
])

display(inventory)


for name, df in tables.items():
    print(f"\n{name}: {df.shape}")

    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (
            100 * df.isna().mean()
        ).round(2),
    })

    display(summary)


,table,file,rows,columns,subscriber_ids,grain_from_documentation
0,account,2_subscriber_account.csv,2110,8,2104.0,one row per subscriber
1,subscription,3_subscription.csv,2104,9,2104.0,one row per subscription
2,viewing,4_viewing_behaviour.csv,8939,11,2104.0,one row per viewing session
3,engagement,5_engagement_behaviour.csv,28118,7,2104.0,one row per engagement event
4,content,5b_content_reference.csv,880,3,NaN,one row per content item
5,billing,6_billing_payment_history.csv,32114,9,2104.0,one row per billing event
6,outcome,7_cancellation_retention.csv,539,11,539.0,one row per cancellation/retention event



account: (2110, 8)


,dtype,missing,missing_pct
subscriber_id,str,0,0.00
household_id,str,0,0.00
account_creation_date,datetime64[us],0,0.00
acquisition_date,datetime64[us],0,0.00
acquisition_channel,str,0,0.00
acquisition_campaign,str,278,13.18
customer_segment,str,0,0.00
province,str,85,4.03



subscription: (2104, 9)


,dtype,missing,missing_pct
subscriber_id,str,0,0.00
subscription_id,str,0,0.00
subscription_plan,str,0,0.00
subscription_start_date,datetime64[us],0,0.00
subscription_end_date,datetime64[us],1727,82.08
subscription_status,str,0,0.00
monthly_fee,float64,0,0.00
discount_amount,float64,0,0.00
promotion_code,str,966,45.91


...

,dtype,missing,missing_pct
content_id,str,0,0.0
title,str,0,0.0
category,str,0,0.0



billing: (32114, 9)


,dtype,missing,missing_pct
subscriber_id,str,0,0.0
billing_date,datetime64[us],0,0.0
amount_charged,float64,0,0.0
payment_status,str,0,0.0
payment_method,str,0,0.0
payment_failure_indicator,bool,0,0.0
payment_failure_count,int64,0,0.0
refund_indicator,bool,0,0.0
refund_amount,float64,0,0.0



outcome: (539, 11)


,dtype,missing,missing_pct
subscriber_id,str,0,0.00
cancellation_date,datetime64[us],0,0.00
cancellation_reason,str,0,0.00
cancellation_channel,str,0,0.00
churn_indicator,bool,0,0.00
reactivation_indicator,bool,0,0.00
reactivation_date,datetime64[us],497,92.21
retention_interaction_date,datetime64[us],134,24.86
retention_offer,str,134,24.86
offer_accepted,object,134,24.86


### Integrity and schema validation

We check the documented keys, duplicates, missing identifiers, date coverage, category cleanliness and basic numeric plausibility. A duplicate is reported first; the six duplicated account rows are exact row-for-row duplicates, so the modelling copy removes only exact duplicates (the raw table remains unchanged). Missing `province`, `promotion_code`, search fields and `subscription_end_date` have documented meanings and are not treated as generic errors.

#### Duplicate and key checks

In [7]:
key_map = {
    "account": ["subscriber_id"],
    "subscription": ["subscriber_id", "subscription_id"],
    "viewing": ["session_id"],
    "content": ["content_id"],
    "outcome": ["subscriber_id"],
}

quality_rows = []

for name, df in tables.items():
    key_columns = key_map.get(name, [])

    quality_rows.append({
        "Table": name,
        "Rows": len(df),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Missing Subscriber IDs": (
            int(df["subscriber_id"].isna().sum())
            if "subscriber_id" in df.columns else np.nan
        ),
        "Key Columns": ", ".join(key_columns),
        "Duplicate Keys": (
            int(df.duplicated(key_columns).sum())
            if key_columns and all(c in df.columns for c in key_columns)
            else np.nan
        ),
    })

display(pd.DataFrame(quality_rows))

for name in ["account", "subscription", "outcome"]:
    print(
        f"{name.capitalize()} subscriber IDs unique: "
        f"{tables[name]['subscriber_id'].is_unique}"
    )

,Table,Rows,Duplicate Rows,Missing Subscriber IDs,Key Columns,Duplicate Keys
0,account,2110,6,0.0,subscriber_id,6.0
1,subscription,2104,0,0.0,"subscriber_id, subscription_id",0.0
2,viewing,8939,0,0.0,session_id,0.0
3,engagement,28118,3,0.0,,NaN
4,content,880,0,NaN,content_id,0.0
5,billing,32114,0,0.0,,NaN
6,outcome,539,0,0.0,subscriber_id,0.0


Account subscriber IDs unique: False
Subscription subscriber IDs unique: True
Outcome subscriber IDs unique: True


#### Data quality checks

In [9]:
date_quality = []

for name, columns in date_cols.items():
    df = tables[name]

    for column in columns:
        date_quality.append({
            "Table": name,
            "Column": column,
            "Parsed / Total": (
                f"{df[column].notna().sum(): } / {len(df): }"
            ),
            "Earliest Date": df[column].min(),
            "Latest Date": df[column].max(),
        })

display(pd.DataFrame(date_quality))

,Table,Column,Parsed / Total,Earliest Date,Latest Date
0,account,account_creation_date,2110 / 2110,2024-01-02 06:34:25,2026-05-20 22:15:38
1,account,acquisition_date,2110 / 2110,2024-01-02 09:39:03,2026-05-21 00:21:00
2,subscription,subscription_start_date,2104 / 2104,2024-01-02 09:39:03,2026-05-21 00:21:00
3,subscription,subscription_end_date,377 / 2104,2024-11-05 09:31:57,2026-06-30 23:59:59
4,viewing,viewing_timestamp,8939 / 8939,2024-01-14 15:01:13,2026-06-29 23:58:33
5,engagement,event_timestamp,28118 / 28118,2024-01-14 14:48:26,2026-06-29 23:57:48
6,billing,billing_date,32114 / 32114,2024-01-02 00:00:00,2026-06-30 00:00:00
7,outcome,cancellation_date,539 / 539,2024-10-31 09:31:57,2026-06-23 17:50:02
8,outcome,reactivation_date,42 / 539,2025-06-26 18:30:59,2026-06-30 10:16:51
9,outcome,retention_interaction_date,405 / 539,2024-10-27 14:38:09,2026-06-22 15:44:24


#### Referential integrity checks

In [10]:
viewing_content_ids = tables["viewing"]["content_id"]
reference_content_ids = tables["content"]["content_id"]

unmatched_content_ids = (
    ~viewing_content_ids.isin(reference_content_ids)
).sum()

print("Viewing content IDs without reference:", int(unmatched_content_ids))

account_ids = set(tables["account"]["subscriber_id"].dropna())

for name in ["subscription", "viewing", "engagement", "billing", "outcome"]:
    subscriber_ids = set(tables[name]["subscriber_id"].dropna())
    unmatched_ids = subscriber_ids - account_ids

    print(f"{name.capitalize()}: subscriber IDs not in account table = {len(unmatched_ids)}")

Viewing content IDs without reference: 0
Subscription: subscriber IDs not in account table = 0
Viewing: subscriber IDs not in account table = 0
Engagement: subscriber IDs not in account table = 0
Billing: subscriber IDs not in account table = 0
Outcome: subscriber IDs not in account table = 0


**Cleaning decisions.** The account table is the subscriber population anchor. `household_id` is not a unique key. The supplied dictionary says `payment_failure_count` is a repeated observation-period total, so it must not be summed across billing rows; this notebook excludes it from event aggregation and instead counts event-level failures from `payment_failure_indicator`. Category text is trimmed and normalised for consistent encoding, but raw source tables are retained unchanged.

In [11]:
# Create modelling copies with consistent whitespace/casing 
# for categorical fields only
CAT_COLS = {
    "account": [
        "acquisition_channel",
        "acquisition_campaign",
        "customer_segment",
        "province",
    ],
    "subscription": [
        "subscription_plan",
        "subscription_status",
        "promotion_code",
    ],
    "viewing": [
        "content_category",
        "device_type",
    ],
    "engagement": [
        "event_type",
    ],
    "billing": [
        "payment_status",
        "payment_method",
    ],
    "outcome": [
        "cancellation_reason",
        "cancellation_channel",
        "retention_offer",
        "offer_outcome",
    ],
}

# IDs and free-text search terms are preserved
# categorical fields are normalised below
for name, columns in CAT_COLS.items():
    df = tables[name]

    for column in columns:
        if column not in df.columns:
            continue

        values = (
            df[column]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.title()
        )

        tables[name][column] = values.mask(
            values.eq(""),
            pd.NA,
        )

print("Categorical columns normalised.")

print(
    "Observed subscription statuses:",
    tables["subscription"]["subscription_status"]
    .value_counts(dropna=False)
    .to_dict(),
)

print(
    "Observed churn labels:",
    tables["outcome"]["churn_indicator"]
    .value_counts(dropna=False)
    .to_dict(),
)


Categorical columns normalised.
Observed subscription statuses: {'Active': 1675, 'Cancelled': 368, 'Suspended': 40, 'Lapsed': 12, 'Reactivated': 9}
Observed churn labels: {True: 368, False: 171}


### Target definition and eligibility

The official label is `churn_indicator` in the cancellation/retention table. We do not reconstruct the 30-day reactivation rule from incomplete assumptions; we validate the supplied flag against the documented cancellation and reactivation fields, then use it as authoritative.

For each month-end index date, an account is eligible if it was acquired by that date and has no recorded cancellation on or before that date. A positive label means a qualifying churn event occurs in `(index date, index date + 30 days]`. An eligible subscriber with no qualifying event in that window is negative. Index dates stop at **31 May 2026** so the 30-day outcome window is covered by the extract ending 30 June 2026. This is a forward-looking event label, not classification of an already-cancelled account.

In [12]:
account = tables["account"].copy().drop_duplicates()  # Repeated rows are exact duplicates.
subscription = tables["subscription"].copy()
outcome = tables["outcome"].copy()

# Validate the supplied target against the key edge-case fields in the dictionary
print(
    "Outcome rows:", len(outcome),
    "| unique subscribers:", outcome["subscriber_id"].nunique()
)

print("Target by reactivation flag:")
display(
    pd.crosstab(
        outcome["reactivation_indicator"],
        outcome["churn_indicator"],
        dropna=False,
    )
)

print("Target by offer outcome:")
display(
    pd.crosstab(
        outcome["offer_outcome"],
        outcome["churn_indicator"],
        dropna=False,
    )
)

observation_start = pd.Timestamp("2024-01-01")
observation_end = pd.Timestamp("2026-06-30 23:59:59")

outside_window = (
    (outcome["cancellation_date"] < observation_start)
    | (outcome["cancellation_date"] > observation_end)
)

print(
    "Cancellation dates outside documented observation window:",
    int(outside_window.sum()),
)

# Fail visibly if the supplied target contradicts explicit dictionary edge cases
contradictions = outcome[
    outcome["offer_outcome"].isin(["Paused", "Retained", "Downgraded"])
    & outcome["churn_indicator"].eq(True)
]

print(
    "Contradictions with non-churn offer outcomes requiring review:",
    len(contradictions),
)

if len(contradictions):
    display(contradictions.head(10))

# The extract describes 1:1 account-subscription relationships but contains six unmatched accounts
account_ids = set(account["subscriber_id"])
subscription_ids = set(subscription["subscriber_id"])

print(
    "Accounts without subscription row:",
    len(account_ids - subscription_ids),
)

print(
    "Subscription rows without account:",
    len(subscription_ids - account_ids),
)

# Parse and validate key timestamps
# Account creation/acquisition may be same-day but should not be after index
for df, column in [
    (account, "acquisition_date"),
    (account, "account_creation_date"),
    (subscription, "subscription_start_date"),
]:
    print(
        column,
        "missing:",
        int(df[column].isna().sum()),
    )

Outcome rows: 539 | unique subscribers: 539
Target by reactivation flag:


churn_indicator,False,True
reactivation_indicator,,
False,162,335
True,9,33


Target by offer outcome:


churn_indicator,False,True
offer_outcome,,
Churned,0,335
Downgraded,36,0
Paused,23,0
Reactivated Later,9,33
Retained,103,0


Cancellation dates outside documented observation window: 0
Contradictions with non-churn offer outcomes requiring review: 0
Accounts without subscription row: 0
Subscription rows without account: 0
acquisition_date missing: 0
account_creation_date missing: 0
subscription_start_date missing: 0


### Experimental design and chronological partitions

Monthly snapshots let the model learn from earlier periods and be evaluated on later periods. The intended scenario is future behaviour of existing subscribers, so the same subscriber can legitimately appear in multiple time partitions; this is not a new-subscriber generalisation test. We report subscriber overlap and treat repeated observations as a limitation when interpreting uncertainty.

Partitions follow the supplied reference periods where possible: training through September 2025, validation October 2025 to January 2026, and test February–May 2026. The last period is possible because the chosen 30-day horizon ends by June 2026. Splits are by **index date**, never random row splitting.

In [13]:
INDEX_DATES = pd.date_range('2024-04-30','2026-05-31',freq='ME')
print('Index dates:', len(INDEX_DATES), '| first:', INDEX_DATES.min().date(), '| last:', INDEX_DATES.max().date())
print('Training index dates: through 2025-09-30; validation: 2025-10-31 to 2026-01-31; test: 2026-02-28 to 2026-05-31.')

Index dates: 26 | first: 2024-04-30 | last: 2026-05-31
Training index dates: through 2025-09-30; validation: 2025-10-31 to 2026-01-31; test: 2026-02-28 to 2026-05-31.


### Focused exploratory analysis

Inspect the target event distribution and the available activity coverage. Event counts are not subscriber counts and this distinction matters because high-activity subscribers generate many rows.

In [14]:
VISUALISATION_DIR = Path.cwd().resolve() / "visualisations"
VISUALISATION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Visualisations directory: {VISUALISATION_DIR}")

Visualisations directory: C:\Users\sltha\Documents\STADIO\CAP182\capstone-project\notebooks\visualisations


#### Recorded Cancellation Outcomes

In [ ]:
churn_counts = (
    outcome["churn_indicator"]
    .value_counts()
    .sort_index()
    .rename(index={False: "Not churn", True: "Churn"})
)

fig, ax = plt.subplots(figsize=(7, 5))

churn_counts.plot(kind="bar", ax=ax, color=["steelblue", "indianred"])

ax.set_title("Recorded Cancellation Outcomes")
ax.set_xlabel("Outcome")
ax.set_ylabel("Number of Subscribers")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
fig.savefig(
    VISUALISATION_DIR / "recorded_cancellation_outcomes.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

#### Timestamped Source-record CCoverage

In [ ]:
date_columns = {
    "Viewing": ("viewing", "viewing_timestamp"),
    "Engagement": ("engagement", "event_timestamp"),
    "Billing": ("billing", "billing_date"),
}

fig, ax = plt.subplots(figsize=(12, 5))

for label, (table_name, column) in date_columns.items():
    dates = tables[table_name][column].dropna()

    monthly_counts = (
        dates.dt.to_period("M")
        .value_counts()
        .sort_index()
    )

    ax.plot(
        monthly_counts.index.astype(str),
        monthly_counts.to_numpy(dtype=float),
        marker="o",
        markersize=3,
        label=label,
    )

ax.set_title("Monthly Source-Record Coverage")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Records")
ax.tick_params(axis="x", rotation=90)
ax.legend(title="Source Table")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(
    VISUALISATION_DIR / "monthly_source_record_coverage.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)